## TL;DR — Plain English

**What this notebook does:**
Teaches you to navigate the OpenFold/Boltz source code like a senior engineer —
finding any component in < 5 minutes, understanding design choices, writing fixes.

**By the end, you will be able to:**
- Navigate OpenFold to find triangle attention, FAPE loss, or data pipeline in < 2 min
- Reproduce any module's forward pass in 20 lines of standalone code
- Explain what `Rigid` and `Rotation` do in `rigid_utils.py`
- Debug a broken backbone frame construction (the most common AF3 coding bug)

**Why this matters:** At Isomorphic Labs, your first task will likely be reading and understanding
existing AF3 code — not writing from scratch. This notebook prepares you for that reality.


# Module 10/00 — OpenFold3 Code Walkthrough
## Navigate the OpenFold Source Code Like an Expert

**What you will learn:** How to read production-grade protein structure prediction code.
After this notebook you can open any file in OpenFold/OpenFold3 and understand what it does
and where it fits in the pipeline.

**Prerequisites:** Modules 07/01-07/04 (architecture concepts). No need to install OpenFold.

**Approach:** We reproduce key logic in ~20-line snippets. For each snippet, we:
1. Show the corresponding OpenFold file/function
2. Explain the design choices
3. Run a minimal working version


## 🟢 Complete Beginner's Guide

**Assumed background:** Comfortable with PyTorch, but has never navigated a large ML research codebase.

### What you need to know before starting

- **Large ML codebases** typically have 10,000–100,000 lines across dozens of files. You cannot read them linearly — you navigate them.
- **Entry points** — start with `train.py`, `run_inference.py`, or `main.py`. These show you what gets called first and give you the top-level flow.
- **Module hierarchy** — code is split into `model/`, `data/`, `utils/`, `config/` etc. Understanding the folder structure tells you where to look for what.
- **Config files** — large codebases use `.yaml` or `.json` configs to separate hyperparameters from code. Read the config to understand what the model is doing.
- **Tracing the forward pass** — follow `model.forward()` step by step; each submodule's `forward()` is the code that runs during inference.

### A 5-step strategy for reading a new ML codebase

1. **Read the README** — understand the paper it implements and the install steps.
2. **Find the entry point** — what file does `python train.py` or the example command call?
3. **Map the model class** — find the top-level `nn.Module` and its `__init__` and `forward`.
4. **Trace one forward pass with dummy data** — print tensor shapes at each layer.
5. **Read the config** — understand every hyperparameter before modifying any code.

### Vocabulary you MUST know

| Term | One-line definition |
|------|--------------------|
| `nn.Module` | PyTorch base class for all models; subclass it to build layers |
| `register_buffer` | Stores a tensor in a module that is not a trainable parameter |
| `state_dict` | Dictionary of all model weights, used for saving/loading |
| `DataParallel / DDP` | Utilities for multi-GPU training |
| `LoRA` | Low-Rank Adaptation — a technique for fine-tuning large models cheaply |

### 3-Step Reading Strategy for Beginners

1. **Read all markdown cells first** — build a mental map of what the codebase does.
2. **Run code cells one at a time** — each cell opens a specific file or runs a specific component.
3. **Modify one function argument and re-run** — change a hidden dimension size and trace how that propagates.

### If you're stuck

- **YouTube:** 'How to Read AI Research Code' by Yannic Kilcher (https://www.youtube.com/watch?v=L-min6ebloc)
- **YouTube:** 'Understanding Large Codebases' by Andrej Karpathy (nanoGPT walkthrough, https://www.youtube.com/watch?v=kCc8FmEb1nY)
- **Book:** *Software Engineering for Data Scientists* — Chapter on navigating existing codebases.
- **Web:** https://github.com/aqlaboratory/openfold — OpenFold GitHub with architecture docs.


---
## Repository Structure Overview

```
openfold/
├── model/
│   ├── model.py               ← Top-level: AlphaFold class, forward() entry point
│   ├── evoformer.py           ← EvoformerStack (AF2); replaced by Pairformer in AF3
│   ├── triangular_attention.py ← TriangleAttentionStartingNode/EndingNode
│   ├── triangular_multiplicative_update.py ← TriangleMultiplicationOutgoing/Incoming
│   ├── pair_transition.py     ← PairTransition (pair MLP)
│   ├── single_transition.py   ← Single representation MLP
│   ├── structure_module.py    ← IPA + angle prediction (AF2); diffusion in AF3
│   ├── embedders.py           ← InputEmbedder, RecyclingEmbedder, ExtraMSAEmbedder
│   └── heads.py               ← DistogramHead, MaskedMSAHead, pLDDTHead
├── utils/
│   ├── loss.py                ← All loss functions (FAPE, distogram, violations...)
│   ├── feats.py               ← Feature processing utilities
│   ├── rigid_utils.py         ← Rigid body transforms (Rotations + Translations)
│   └── geometry.py            ← 3D geometry utilities
├── data/
│   ├── data_pipeline.py       ← Full data processing: PDB → model input tensors
│   ├── mmcif_parsing.py       ← Parse mmCIF files → Python structures
│   ├── templates.py           ← Template feature computation
│   └── feature_pipeline.py   ← Feature post-processing, crop, pad
└── config.py                  ← All hyperparameters in one place
```

### Reading strategy
1. Start at `model/model.py:AlphaFold.forward()` — this is the entry point
2. Follow the data flow through each module
3. When you see an unfamiliar operation, find it in `utils/`
4. Check `config.py` for the hyperparameter values


In [ ]:
# FILE 1: rigid_utils.py — The Foundation of All 3D Operations
# OpenFold path: openfold/utils/rigid_utils.py
# Key classes: Rotation, Rigid
# Why this matters: EVERY frame transformation in AF3 uses these classes

import torch
import numpy as np

print("RIGID BODY TRANSFORMS — The language of protein geometry")
print("=" * 60)
print()
print("A protein frame = (rotation R, translation t)")
print("  R: 3x3 orthonormal matrix  (where am I pointing?)")
print("  t: 3D vector               (where am I located?)")
print()
print("OpenFold represents this as the Rigid class:")
print("  Rigid(Rotation(rot_mats), Trans(trans_vecs))")
print()

class SimpleRotation:
    # Simplified version of openfold.utils.rigid_utils.Rotation
    def __init__(self, rot_mats):
        # rot_mats: (..., 3, 3)
        self.rot_mats = rot_mats

    def compose(self, other):
        # R_combined = R1 @ R2
        return SimpleRotation(self.rot_mats @ other.rot_mats)

    def apply(self, pts):
        # Apply rotation to points: R @ pts
        # pts: (..., 3)
        return torch.einsum('...ij,...j->...i', self.rot_mats, pts)

    def invert(self):
        # R^-1 = R^T for rotation matrices
        return SimpleRotation(self.rot_mats.transpose(-1, -2))

class SimpleRigid:
    # Simplified version of openfold.utils.rigid_utils.Rigid
    def __init__(self, rots, trans):
        self.rots = rots
        self.trans = trans  # (..., 3)

    def apply(self, pts):
        # T(x) = R @ x + t
        return self.rots.apply(pts) + self.trans

    def invert_apply(self, pts):
        # T^-1(x) = R^T @ (x - t)
        return self.rots.invert().apply(pts - self.trans)

    def compose(self, other):
        # (R1, t1) compose (R2, t2) = (R1@R2, R1@t2 + t1)
        new_rots = self.rots.compose(other.rots)
        new_trans = self.rots.apply(other.trans) + self.trans
        return SimpleRigid(new_rots, new_trans)

# Test: build a frame from N, CA, C backbone atoms (gram-schmidt)
def backbone_frame(n_pos, ca_pos, c_pos):
    # Standard backbone frame definition (same as openfold/utils/feats.py:atom14_to_frames)
    v1 = c_pos - ca_pos      # CA → C direction
    v2 = n_pos - ca_pos      # CA → N direction

    # Gram-Schmidt orthonormalization
    e1 = v1 / (v1.norm(dim=-1, keepdim=True) + 1e-8)
    u2 = v2 - (v2 * e1).sum(-1, keepdim=True) * e1
    e2 = u2 / (u2.norm(dim=-1, keepdim=True) + 1e-8)
    e3 = torch.cross(e1, e2, dim=-1)

    rot = torch.stack([e1, e2, e3], dim=-1)   # (L, 3, 3) where columns are basis vectors
    return SimpleRigid(SimpleRotation(rot), ca_pos)

# Build frames for a small protein
L = 5
n_pos  = torch.randn(L, 3) * 5
ca_pos = torch.randn(L, 3) * 5
c_pos  = torch.randn(L, 3) * 5

frames = backbone_frame(n_pos, ca_pos, c_pos)
test_pt = torch.randn(3)

# Round-trip: apply frame then invert → original point
pt_in_frame = frames.invert_apply(ca_pos)
pt_back = frames.apply(pt_in_frame)

print(f"CA positions: {ca_pos.shape}")
print(f"Round-trip error: {(pt_back - ca_pos).norm(dim=-1).max():.2e}  (should be ~0)")
print()
print("OpenFold code equivalent:")
print("  from openfold.utils.rigid_utils import Rigid, Rotation")
print("  frames = Rigid(Rotation(rot_mats=rot), trans=ca_pos)")
print("  local_pts = frames.invert_apply(atom_positions)")

---
## The Pairformer Stack

### Where to Find It
- **AF2 (Evoformer):** `openfold/model/evoformer.py` — `EvoformerStack`
- **AF3 (Pairformer):** In OpenFold3 forks: `openfold/model/pairformer.py`

### Key Difference from AF2
AF2's Evoformer updates BOTH the MSA representation AND the pair representation.
AF3's Pairformer removes the MSA update — it only maintains pair + single representations.
This makes it ~3x faster per block but loses direct MSA-pair interaction.

AF3 compensates with the **OuterProductMean** operation that projects MSA → pair once
at the beginning, rather than repeatedly throughout the stack.


In [ ]:
# FILE 2: triangular_attention.py
# OpenFold path: openfold/model/triangular_attention.py
# Key class: TriangleAttentionStartingNode, TriangleAttentionEndingNode

print("TRIANGLE ATTENTION — The Core Pairformer Operation")
print("=" * 60)
print()
print("Intuition: update edge (i,k) by attending over all intermediate nodes j")
print("  z[i,k] = sum_j attention(z[i,j], z[j,k]) * value(z[j,k])")
print()
print("Two variants:")
print("  Starting node: index over i  → update z[i,k] using {z[i,j], z[j,k]}")
print("  Ending node:   index over k  → update z[i,k] using {z[i,j], z[j,k]}")
print()

import torch
import torch.nn as nn
import torch.nn.functional as F

class TriangleAttentionStartingNode(nn.Module):
    # Simplified version of openfold.model.triangular_attention.TriangleAttentionStartingNode
    # Full version: chunk-aware, uses pair bias, has gating
    def __init__(self, c_z=32, c_hidden=8, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.c_per_head = c_hidden // n_heads
        self.norm = nn.LayerNorm(c_z)
        # Projections
        self.q_proj = nn.Linear(c_z, c_hidden, bias=False)
        self.k_proj = nn.Linear(c_z, c_hidden, bias=False)
        self.v_proj = nn.Linear(c_z, c_hidden, bias=False)
        self.b_proj = nn.Linear(c_z, n_heads, bias=False)  # pair bias
        self.g_proj = nn.Linear(c_z, c_hidden)             # gating
        self.out_proj = nn.Linear(c_hidden, c_z)

    def forward(self, z):
        # z: (B, L, L, c_z)
        B, L, _, c_z = z.shape
        z_norm = self.norm(z)

        Q = self.q_proj(z_norm)  # (B, L, L, c_hidden)
        K = self.k_proj(z_norm)
        V = self.v_proj(z_norm)

        # Pair bias: b[i,k,h] = linear(z[i,k]) — each pair biases the attention
        b = self.b_proj(z_norm)  # (B, L, L, n_heads)

        # Reshape to (B, L, n_heads, L, c_per_head) for starting-node attention
        h = self.n_heads
        d = self.c_per_head
        Q = Q.view(B, L, L, h, d).permute(0, 1, 3, 2, 4)  # (B, L, h, L, d) — row i, head h
        K = K.view(B, L, L, h, d).permute(0, 1, 3, 2, 4)
        V = V.view(B, L, L, h, d).permute(0, 1, 3, 2, 4)

        # Attention scores: q[i,h,j] @ k[i,h,k]^T
        # For starting node: for each row i, attend over columns j using query z[i,j]
        scores = torch.einsum('bihkd,bihjd->bihkj', Q, K) / (d ** 0.5)
        b_exp = b.permute(0, 1, 3, 2).unsqueeze(-1)  # (B, L, h, L, 1)
        scores = scores + b_exp.squeeze(-1).unsqueeze(-2)

        attn = F.softmax(scores, dim=-1)  # (B, L, h, L, L)

        # Apply to values
        out = torch.einsum('bihkj,bihjd->bihkd', attn, V)  # (B, L, h, L, d)
        out = out.permute(0, 1, 3, 2, 4).contiguous().view(B, L, L, -1)  # (B, L, L, c_hidden)

        # Gate
        g = torch.sigmoid(self.g_proj(z_norm))
        out = g * out

        return z + self.out_proj(out)  # residual connection

# Test
B, L, c_z = 1, 20, 32
z = torch.randn(B, L, L, c_z)
tri_attn = TriangleAttentionStartingNode(c_z=c_z, c_hidden=16, n_heads=4)
z_out = tri_attn(z)

print(f"Input z:  {z.shape}")
print(f"Output z: {z_out.shape}")
print(f"Parameters: {sum(p.numel() for p in tri_attn.parameters()):,}")
print()
print("OpenFold code equivalent (exact):")
print("  from openfold.model.triangular_attention import TriangleAttentionStartingNode")
print("  block = TriangleAttentionStartingNode(c_z=128, c_hidden=32, no_heads=4)")
print("  z = block(z, mask=pair_mask, chunk_size=None)")

In [ ]:
# FILE 3: loss.py — Where All Training Signals Live
# OpenFold path: openfold/utils/loss.py
# Key functions: fape_loss, distogram_loss, lddt_loss, violation_loss

print("LOSS FUNCTIONS IN OPENFOLD — Key Implementation Details")
print("=" * 60)
print()

print("1. FAPE LOSS (openfold/utils/loss.py:backbone_loss)")
print("-" * 50)
print("   OpenFold signature:")
print("   backbone_loss(backbone_rigid_tensor, backbone_rigid_mask,")
print("                 traj, use_clamped_fape=False, clamp_distance=10.0,")
print("                 loss_unit_distance=10.0)")
print()
print("   Key design choices in OpenFold:")
print("   - backbone_rigid_tensor: (B, T, L, 7) — 4 quaternion + 3 translation")
print("     T = number of recycling/structure module iterations")
print("   - Auxiliary loss on intermediate predictions (not just final)")
print("   - loss_unit_distance normalizes to [0,1] range")
print()

print("2. DISTOGRAM LOSS (openfold/utils/loss.py:distogram_loss)")
print("-" * 50)
print("   def distogram_loss(logits, pseudo_beta, pseudo_beta_mask,")
print("                      min_bin=2.3125, max_bin=21.6875, no_bins=64)")
print()
print("   pseudo_beta: Cb position (or Ca for Gly) — this is the KEY detail")
print("   AF3/AF2 uses Cbeta not Calpha for distance computation:")
print()

# Compute pseudo-Cbeta from backbone atoms
def pseudo_beta(ca_pos, c_pos, n_pos, is_gly=None):
    # For non-glycine: estimate Cb from backbone geometry
    # For glycine: use Ca (no Cb)
    # Uses the virtual Cb trick from the AF paper
    b = ca_pos - n_pos
    c = c_pos - ca_pos
    a = torch.cross(b, c, dim=-1)

    # Cb ≈ CA + (-0.58273431*a + 0.56802827*b - 0.54067466*c)
    cb = ca_pos + (-0.58273431 * a + 0.56802827 * b - 0.54067466 * c)

    if is_gly is not None:
        # Glycine has no Cb → use Ca
        return torch.where(is_gly.unsqueeze(-1), ca_pos, cb)
    return cb

L = 10
ca = torch.randn(L, 3)
c_  = ca + torch.randn(L, 3) * 0.3
n_  = ca + torch.randn(L, 3) * 0.3
is_gly = torch.zeros(L, dtype=torch.bool)
is_gly[3] = True  # residue 3 is glycine

cb = pseudo_beta(ca, c_, n_, is_gly)
print(f"   CA positions: {ca.shape}")
print(f"   Pseudo-CB positions: {cb.shape}")
print(f"   Gly residue 3: CB→CA (same as CA)")
print(f"   CB==CA for Gly: {torch.allclose(cb[3], ca[3])}")
print()

print("3. lDDT PREDICTION LOSS (openfold/utils/loss.py:lddt_loss)")
print("-" * 50)
print("   lDDT is computed between predicted and true ALL-ATOM distances")
print("   Then MSE loss: || predicted_plddt_logits - true_lddt ||")
print("   AF3 uses 50 bins (each = 0.02 lDDT units)")
print()

print("4. VIOLATION LOSS (openfold/utils/loss.py:find_structural_violations)")
print("-" * 50)
print("   Checks: bond length, bond angle, clash (residue-level)")
print("   Returns dict with per-residue violation masks + loss values")
print("   Typical weight in total loss: 0.02-0.1 (regularizer, not primary)")

In [ ]:
# FILE 4: data_pipeline.py — How PDB Files Become Model Inputs
# OpenFold path: openfold/data/data_pipeline.py, mmcif_parsing.py

print("DATA PIPELINE — PDB/mmCIF → Model Input Tensors")
print("=" * 60)
print()
print("Full pipeline (openfold/data/data_pipeline.py:DataPipeline.process_pdb):")
print()
print("  Step 1: Parse mmCIF file")
print("    mmcif_parsing.parse(file_id, mmcif_string)")
print("    → MmcifObject with: seqres, atom_positions, atom_mask, residue_index")
print()
print("  Step 2: Make sequence features")
print("    make_sequence_features(sequence, description, num_res)")
print("    → aatype (one-hot), residue_index, seq_length")
print()
print("  Step 3: Run MSA search (HHblits / JackHMMER)")
print("    queries UNIREF90 (~100GB) → HHblits output → parsed to a3m format")
print("    In practice for fine-tuning: use precomputed MSA or ESM-2 embeddings")
print()
print("  Step 4: Template search (HHsearch)")
print("    search PDB70 database → top 4 templates → template featurizer")
print("    Can be disabled: no_templates=True in config")
print()
print("  Step 5: Feature postprocessing")
print("    feature_pipeline.py:FeaturePipeline.process_features_dict()")
print("    → crop to max_sequence_length")
print("    → generate atom14 coordinates and mask")
print("    → generate ground-truth frames from backbone atoms")
print()

# Simulate the key transformation: sequence → aatype one-hot
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY-'
AA_TO_INT = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def sequence_to_aatype(sequence, unknown_idx=20):
    # Returns int tensor of shape (L,) — same as OpenFold
    return torch.tensor([AA_TO_INT.get(aa, unknown_idx) for aa in sequence])

def aatype_to_onehot(aatype, n_types=21):
    # Returns (L, n_types) float tensor — same as OpenFold
    return torch.nn.functional.one_hot(aatype, n_types).float()

seq = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEFGLAPFLPDQIHFVHSQELLSRYPDLDAKGRERAIAKDLGAVFLVGIGGKLSDGHRHDVRAPDYDDWSTPSELGHAGLNGDILVWNPVLEDAFELSSMGIRVDADTLKHQLALTGDEDRLELEWHQALLRGEMPQTIGGGIGQSRLTMLLLQLPHIGQVQAGVWPAAVRESVPSLL"
aatype = sequence_to_aatype(seq)
onehot = aatype_to_onehot(aatype)

print(f"Example protein: {len(seq)} residues")
print(f"  aatype:  {aatype.shape}, dtype={aatype.dtype}")
print(f"  onehot:  {onehot.shape}, dtype={onehot.dtype}")
print(f"  First 5 aa: {seq[:5]} → {aatype[:5].tolist()}")
print()
print("OpenFold code equivalent:")
print("  from openfold.data.data_transforms import make_seq_mask, make_one_hot")
print("  aatype = torch.tensor([residue_constants.restype_order[aa] for aa in seq])")

In [ ]:
# FILE 5: config.py — All Hyperparameters in One Place
# OpenFold path: openfold/config.py

print("CONFIGURATION — Hyperparameters Reference")
print("=" * 60)
print()

# Key AF3 configuration values (from paper + OpenFold implementation)
config_summary = {
    "Model Architecture": {
        "c_s (single repr dim)": 384,
        "c_z (pair repr dim)": 128,
        "n_pairformer_blocks": 48,
        "n_heads_triangle_attn": 4,
        "c_hidden_triangle_attn": 32,
        "c_hidden_pair_trans": 512,
        "max_recycling_iters": 3,
    },
    "Diffusion Module": {
        "n_timesteps": 200,
        "n_diff_heads": 8,
        "c_diff_token_single": 384,
        "c_diff_pair": 128,
        "n_diff_transformer_layers": 24,
    },
    "Data": {
        "max_sequence_length": 5120,
        "n_extra_msa": 1024,
        "n_msa_cluster": 512,
        "n_templates": 4,
        "n_distogram_bins": 64,
        "n_plddt_bins": 50,
    },
    "Training": {
        "batch_size": 1,
        "gradient_accumulation_steps": 128,
        "learning_rate": 1.8e-3,
        "warmup_steps": 1000,
        "total_training_steps": 50000,
        "weight_decay": 0.0,
        "adam_eps": 1e-8,
        "grad_clip_norm": 0.1,
    },
    "Loss Weights": {
        "w_fape_backbone": 0.5,
        "w_fape_all_atom": 1.0,
        "w_distogram": 0.3,
        "w_masked_msa": 2.0,
        "w_plddt": 0.01,
        "w_violations": 1.0,
    }
}

for section, params in config_summary.items():
    print(f"  [{section}]")
    for k, v in params.items():
        print(f"    {k:<40} = {v}")
    print()

print("OpenFold code equivalent:")
print("  from openfold.config import model_config")
print("  cfg = model_config('model_3')   # AF2 model 3")
print("  cfg.model.evoformer_stack.c_s  # = 384")
print("  cfg.model.heads.distogram.no_bins  # = 64")

In [ ]:
# NAVIGATING OPENFOLD — Practical Guide
print("HOW TO NAVIGATE THE OPENFOLD CODEBASE")
print("=" * 60)
print()

navigation_guide = [
    ("You want to understand...", "Start here"),
    ("-" * 40, "-" * 25),
    ("How a protein sequence becomes input tensors",
     "data/data_pipeline.py:DataPipeline.process_pdb"),
    ("How MSA features are computed",
     "data/data_pipeline.py:_process_msa_feats"),
    ("How templates are processed",
     "data/templates.py:TemplateHitFeaturizer"),
    ("The main model forward pass",
     "model/model.py:AlphaFold.forward"),
    ("Pairformer / Evoformer block details",
     "model/evoformer.py:EvoformerStack.forward"),
    ("Triangle attention (the hard part)",
     "model/triangular_attention.py:TriangleAttentionStartingNode"),
    ("Triangle multiplicative update",
     "model/triangular_multiplicative_update.py"),
    ("How 3D coordinates are predicted (AF2)",
     "model/structure_module.py:InvariantPointAttention"),
    ("How pLDDT is computed from model output",
     "model/heads.py:PerResidueLDDTCaPredictor"),
    ("Rigid body math (frames)",
     "utils/rigid_utils.py:Rigid, Rotation, Quaternion"),
    ("FAPE loss implementation",
     "utils/loss.py:backbone_loss, sidechain_loss"),
    ("Violation checks",
     "utils/loss.py:find_structural_violations"),
    ("How atom positions → backbone frames",
     "utils/feats.py:atom37_to_frames"),
]

for item, loc in navigation_guide:
    if item.startswith('-'):
        print(f"  {item}")
    else:
        print(f"  {item:<45} → {loc}")

print()
print("WORKFLOW FOR A NEW TASK:")
print()
print("1. Find the concept in the architecture paper (AF3 supplementary)")
print("2. Search for the exact class/function name in openfold/")
print("   grep -r 'TriangleAttention' openfold/model/")
print("3. Read the class, noting:")
print("   - Input shapes (always in comments or docstring)")
print("   - Which config params it uses (search config.py for the key)")
print("   - Which utils/ functions it calls")
print("4. Run a small example (like cells in this notebook)")
print("5. Check the unit tests: tests/ directory")

In [ ]:
# EXERCISE: Trace a Mini-Forward Pass
print("EXERCISE — Trace One Input Through Mini-OpenFold")
print("=" * 60)
print()
print("Challenge: implement the minimal AF3-like forward pass")
print("covering every step from raw sequence to predicted coords.")
print("Expected time: 20-30 minutes.")
print()

import torch
import torch.nn as nn

# STEP 1: Input
seq = "ACDEFGHIKL"  # 10 residues
L = len(seq)
AA = 'ACDEFGHIKLMNPQRSTVWY-'
aatype = torch.tensor([AA.index(a) for a in seq])

print(f"Input sequence: {seq} (L={L})")
print(f"aatype: {aatype.tolist()}")

# STEP 2: Embed
c_s, c_z = 16, 8
embed_s = nn.Embedding(21, c_s)
s = embed_s(aatype)  # (L, c_s) — single representation
print(f"Single repr s: {s.shape}")

# STEP 3: Build pair repr via outer sum
z = s.unsqueeze(1) + s.unsqueeze(0)  # (L, L, c_s) — simple outer sum
z_proj = nn.Linear(c_s, c_z)
z = z_proj(z)  # (L, L, c_z) — pair representation
print(f"Pair repr z: {z.shape}")

# STEP 4: One Pairformer block (simplified: pair transition only)
pair_trans = nn.Sequential(
    nn.LayerNorm(c_z), nn.Linear(c_z, c_z * 2), nn.ReLU(), nn.Linear(c_z * 2, c_z)
)
z = z + pair_trans(z)

# STEP 5: Read off predictions
# Distogram head
dist_head = nn.Linear(c_z, 64)
dist_logits = dist_head(z)      # (L, L, 64)

# Coord head (from single repr)
coord_head = nn.Sequential(nn.LayerNorm(c_s), nn.Linear(c_s, 3))
coords = coord_head(s)          # (L, 3)

print(f"Distogram logits: {dist_logits.shape}")
print(f"Predicted coords: {coords.shape}")
print()
print("This covers the full forward pass skeleton.")
print("Real AF3 additions:")
print("  - 48 Pairformer blocks (not 1)")
print("  - Triangle attention in each block (not just pair transition)")
print("  - Diffusion module (not direct coordinate regression)")
print("  - MSA processing before first block")
print("  - Recycling (3x full forward pass)")
print()
print("Each addition is in a separate file you can read in isolation.")
print("Start with model/model.py and follow the .forward() call chain.")

---
## Summary: You Can Now Navigate OpenFold

### Key Files by Topic

| What you need | File |
|---|---|
| Entry point | `model/model.py` |
| Triangle attention | `model/triangular_attention.py` |
| Rigid frames | `utils/rigid_utils.py` |
| All losses | `utils/loss.py` |
| Data pipeline | `data/data_pipeline.py` |
| Config (all hyperparms) | `config.py` |

### Reading Strategy
1. **Always start with shapes** — every function has `# (B, L, c_z)` style comments
2. **Find the config key** — if you see `c_hidden = c.c_hidden_tri_att`, search `c_hidden_tri_att` in config.py
3. **Run it small** — reproduce any function with L=10, c_z=16, B=1 to verify understanding
4. **Use grep** — `grep -r 'class Triangle' openfold/` finds all triangle modules

### What to Read Next

For AF3 specifically (differs from AF2 OpenFold):
1. **AF3 paper:** "Accurate structure prediction of biomolecular interactions with AlphaFold 3" — supplementary methods sections 1-6
2. **Boltz-1/2 codebase:** Open-source AF3 reimplementation — more accessible than Google's
3. **OpenFold3 PR tracker:** Track which AF3 features have been merged into OpenFold

### Now proceed to Module 10/01
You have the conceptual and navigational tools.
Module 10/01 will apply all of this to fine-tune on SKEMPI v2 for ΔΔG prediction.


---
## 📚 Resources — Reading Production ML Code

### University Resources for Code Navigation
| Resource | What You Learn |
|----------|----------------|
| **[MIT 6.S081 Operating Systems](https://pdos.csail.mit.edu/6.828/2022/schedule.html)** | How to read large codebases; labs involve navigating xv6 OS |
| **[Stanford CS110 Principles of CS Systems](https://web.stanford.edu/class/cs110/)** | Code architecture; how to read unfamiliar large repos |
| **[fast.ai Code First approach](https://course.fast.ai/)** | How Jeremy Howard navigates PyTorch internals live on camera |

### OpenFold / AF3 Codebases (Priority Order)
| Repo | Why Read | Start At |
|------|----------|---------|
| **[aqlaboratory/openfold](https://github.com/aqlaboratory/openfold)** | Most readable; PyTorch; best documented | `openfold/model/model.py` |
| **[jwohlwend/boltz](https://github.com/jwohlwend/boltz)** | AF3 open implementation; clean code | `src/boltz/model/model.py` |
| **[google-deepmind/alphafold](https://github.com/google-deepmind/alphafold)** | Original AF2; JAX; harder to read | `alphafold/model/model.py` |
| **[microsoft/protein-frame-flow](https://github.com/microsoft/protein-frame-flow)** | Diffusion for proteins; cleaner than AF3 | `model/flow_module.py` |

### How to Read a Large ML Codebase (MIT Method)
```bash
# Step 1: Understand the entry point
grep -r "def forward" openfold/model/model.py

# Step 2: Map the data flow
grep -r "class.*Module" openfold/model/ | grep -v "test" | sort

# Step 3: Find the loss functions
grep -r "def.*loss" openfold/utils/loss.py | grep "^def"

# Step 4: Find the config
grep -r "c_z\|c_s\|n_heads" openfold/config.py | head -20

# Step 5: Find the tests
ls openfold/tests/  # unit tests show expected inputs/outputs
```

### Essential Concepts from `rigid_utils.py`
- **Rigid body transforms**: every protein coordinate operation uses `Rigid` class
- **Quaternion vs rotation matrix**: OpenFold uses both; quaternions for gradients, matrices for operations
- **Frame application**: `rigid.apply(pts)` = R @ pts + t
- **Frame inversion**: `rigid.invert_apply(pts)` = R^T @ (pts - t) — this is FAPE's local frame transform

### Reading Strategy for AF3 Newcomers
1. **Week 1**: Read `model.py:AlphaFold.forward()` — understand the high-level flow
2. **Week 2**: Read `rigid_utils.py` — understand Rigid, Rotation. Run the tests.
3. **Week 3**: Read `triangular_attention.py` — understand triangle attention shapes and chunking
4. **Week 4**: Read `loss.py:backbone_loss` — understand FAPE implementation
5. **Week 5**: Read `data_pipeline.py` — understand the full data processing chain
6. **Week 6**: Make a small change (add a print statement, change a hyperparameter) and verify behavior

### Interview Questions — Codebase Navigation

**Q1:** Given `openfold/model/triangular_attention.py`, how would you find the chunking implementation?
> **A:** `grep -n "chunk" triangular_attention.py` — finds all references. Look for `chunk_size` parameter and the for loop that processes Z[start:end].

**Q2:** How does the OpenFold config system work?
> **A:** `config.py` defines nested dicts/objects with all hyperparameters. Each module reads its relevant sub-config: `c = config.model.evoformer_stack`. This pattern separates architecture from hyperparameters — you can change N_heads without editing the module code.

**Q3:** You find a bug in `loss.py:fape_loss` where clamping is applied before averaging. How do you fix it and verify?
> **A:** Fix the order in the source code, then run the corresponding unit test: `pytest openfold/tests/test_loss.py::test_fape`. If no test exists, write one with a case where the bug would change output (e.g., a coordinate > d_clamp).


---
## 🔧 Debug Exercises — Reading and Fixing OpenFold-Style Code



In [ ]:
# DEBUG EXERCISE: Broken backbone frame construction
# This is a simplified version of openfold/utils/feats.py:atom37_to_frames
# There are 2 bugs — find them

import torch

def backbone_frame_buggy(n_pos, ca_pos, c_pos):
    # Build rotation matrix from backbone atoms (N, CA, C)
    # n_pos, ca_pos, c_pos: (L, 3) — one per residue
    v1 = c_pos - ca_pos        # CA → C
    v2 = n_pos - ca_pos        # CA → N

    # Bug 1: wrong normalization — what's missing?
    e1 = v1                                  # should be v1 / ||v1||

    # Gram-Schmidt: remove e1 component from v2
    u2 = v2 - (v2 * e1).sum(-1, keepdim=True) * e1
    e2 = u2 / (u2.norm(dim=-1, keepdim=True) + 1e-8)

    e3 = torch.cross(e1, e2, dim=-1)

    # Bug 2: wrong rotation matrix construction — should columns be basis vectors?
    rot = torch.stack([e1, e2, e3], dim=-2)  # BUG: should be dim=-1 (columns, not rows)
    return rot

def backbone_frame_correct(n_pos, ca_pos, c_pos):
    v1 = c_pos - ca_pos
    v2 = n_pos - ca_pos

    # Normalize e1 first (Bug 1 fix)
    e1 = v1 / (v1.norm(dim=-1, keepdim=True) + 1e-8)

    u2 = v2 - (v2 * e1).sum(-1, keepdim=True) * e1
    e2 = u2 / (u2.norm(dim=-1, keepdim=True) + 1e-8)
    e3 = torch.cross(e1, e2, dim=-1)

    # Columns are basis vectors (Bug 2 fix)
    rot = torch.stack([e1, e2, e3], dim=-1)  # (L, 3, 3) — columns = [e1, e2, e3]
    return rot

L = 10
ca = torch.randn(L, 3)
n_ = ca + torch.randn(L, 3) * 0.3
c_ = ca + torch.randn(L, 3) * 0.3

R_buggy = backbone_frame_buggy(n_, ca, c_)
R_correct = backbone_frame_correct(n_, ca, c_)

# A valid rotation matrix must have R^T R = I (orthonormality)
I_buggy = R_buggy @ R_buggy.transpose(-1, -2)
I_correct = R_correct @ R_correct.transpose(-1, -2)

eye = torch.eye(3).unsqueeze(0).expand(L, -1, -1)
err_buggy = (I_buggy - eye).norm()
err_correct = (I_correct - eye).norm()

print(f"Buggy R^T R = I error:   {err_buggy:.4f}  (should be ~0 for valid rotation)")
print(f"Correct R^T R = I error: {err_correct:.6f}  (near-zero)")
print()
print("LESSON:")
print("  Bug 1: unnormalized e1 means e1 has length != 1 → not a rotation matrix")
print("  Bug 2: stacking along rows puts basis vectors as rows, not columns")
print("         In OpenFold convention, rotation matrix columns = basis vectors")
print("         R @ v = 'express v in frame coordinates' requires columns = frame axes")

## ✅ Mastery Check — OpenFold3 Codebase Walkthrough

_Answer these questions to verify your understanding._

**Q1 (Recall):** In the OpenFold codebase, which file defines the top-level model class? What is the name of the main `nn.Module` subclass that implements the full AF2/AF3 model?

**Q2 (Understand):** The codebase uses `torch.utils.checkpoint` in several places. What does this do, and why is it necessary for training large structure prediction models?

**Q3 (Apply):** You want to add a new MSA processing layer between the input embedding and the Evoformer stack. Identify the exact file and function where you would insert this, and describe what tensor shapes it must accept and output.

**Q4 (Analyze):** The codebase separates `AlphaFoldConfig` from model code. Why is this separation architecturally important when running ablation studies?

**Q5 (Design):** You need to fine-tune only the Pairformer (pair representation layers) for a TCR-pMHC binding task while freezing all other weights. Write the PyTorch code snippet that sets `requires_grad=False` for all parameters except those in the pairformer module.

---
**Self-assessment rubric:**
- Q1–Q3: ready to move to Module 10/01 (Protein Structure Fine-Tuning capstone)
- Q1–Q4: strong understanding of production-grade ML codebases
- Q1–Q5: interview-ready for ML research engineer roles requiring codebase ownership
